# JobVista — pandas, rating nhu cầu và khoảng cách (Lab 3)

Bản bổ sung theo góp ý giảng viên ngày 05/11/2026. Nguồn TiniX AI có 606.878
bản ghi; catalog đã lọc có **7.050 tin**, giữ 500 ID cũ. Dữ liệu lịch sử, chưa
xác minh còn tuyển. Rating trong ví dụ là **dữ liệu mô phỏng**, không phải nhu cầu thật.

Nguồn: https://huggingface.co/datasets/tinixai/vietnamese-job-descriptions

Snapshot: `679c3a17347ff6ce769d9bf510c122a087493cf6`, CC BY-NC 4.0.

Tọa độ: https://download.geonames.org/export/dump/ (GeoNames, CC BY 4.0).
Xem `docs/LAB3_RATING_DISTANCE.md` để trích dẫn và giải thích giới hạn.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data/jobs_catalog.csv').exists():
    raise FileNotFoundError('Mở notebook từ thư mục project.')
sys.path.insert(0, str(ROOT))
from demand_rating import DemandRatingStore, weighted_rating
from job_proximity import home_locations, compare_options
from catalog_data import localize_jobs
from hybrid_recommender import HybridJobRecommenderSystem

manifest = json.loads((ROOT/'data/catalog_manifest.json').read_text(encoding='utf-8'))
display(pd.Series({key:manifest[key] for key in ['source_rows_scanned','prefiltered_rows','selected_rows','added_rows','company_count']}))

source_rows_scanned    606878
prefiltered_rows        14373
selected_rows            7050
added_rows               6550
company_count            3594
dtype: int64

## 1. pandas đọc theo chunks và làm sạch

Pipeline nguồn dùng Parquet batch4096, pandas vector masks/drop_duplicates rồi NLP.
Ở đây minh họa đọc CSV đã sạch theo chunks, ép kiểu số, xử lý thiếu và bỏ trùng ID.
Không nạp toàn bộ 606.878 mô tả gốc vào notebook.

In [2]:
chunks = []
for batch in pd.read_csv(ROOT/'data/jobs_catalog.csv', chunksize=2048, encoding='utf-8-sig'):
    batch['Experience_Years'] = pd.to_numeric(batch['Experience_Years'], errors='coerce')
    batch = batch.dropna(subset=['Job_ID','Title','Experience_Years'])
    chunks.append(batch)
jobs = pd.concat(chunks, ignore_index=True).drop_duplicates('Job_ID').fillna('')
for column in ['Domain','Role_Code','Primary_Region']:
    jobs[column] = jobs[column].astype('category')
assert len(jobs) == manifest['selected_rows'] == 7050
display(jobs[['Job_ID','Title','Role_Code','Primary_Region','Experience_Years']].head())
display(jobs.groupby('Role_Code', observed=True).agg(jobs=('Job_ID','count'), companies=('Company','nunique')).sort_values('jobs',ascending=False))

,Job_ID,Title,Role_Code,Primary_Region,Experience_Years
0,VN0001,Fresher Product Owner,software,ha_noi,0.0
1,VN0002,Fullstack Java Developer,fullstack,ha_noi,2.0
2,VN0003,Lập Trình Nhúng-Embedded Software Engineer (C/...,embedded,ha_noi,4.0
3,VN0004,Backend Developer (NodeJS),backend,ha_noi,3.0
4,VN0005,Software Quality Assurance (SQA) Engineer,qa,ha_noi,1.0


,jobs,companies
Role_Code,,
software,2537,1706
support,519,401
backend,497,394
ba,462,385
fullstack,441,334
frontend,374,311
data_analyst,365,275
qa,347,302
mobile,285,215


## 2. Rating ngầm: ví dụ 8 trong 10

Một mã trình duyệt chỉ đóng góp nguyện vọng gần nhất; nhiều nghề chia đều 1 lượt.
Pandas `sort_values/drop_duplicates/explode/groupby/merge` tương ứng cách làm Lab 3.
`R = 5*n/N`, `v=N`, `C=5/K`, `m=5`; `WR=v/(v+m)*R+m/(v+m)*C`.

WR đo quan tâm nghề, không đo chất lượng công việc. Ví dụ được tính trong RAM;
**không ghi bất cứ dữ liệu mô phỏng nào vào `.local/jobvista.sqlite3`**.

In [3]:
# Dữ liệu minh họa, không phải người dùng thực.
events = pd.DataFrame([{'actor':f'demo-{i}',
                        'roles':['data_analyst' if i < 8 else 'backend'],
                        'updated_at':i} for i in range(10)])
snapshot = DemandRatingStore.aggregate(events)
ratings = pd.DataFrame(snapshot['roles'])
display(ratings.loc[ratings.demand_count > 0, ['role_code','demand_count','actors','vote_count','vote_average','rating']])
da = ratings.set_index('role_code').loc['data_analyst']
assert da.demand_count == 8 and da.vote_average == 4
assert abs(da.rating - weighted_rating(4,10,5,snapshot['C'])) < 1e-10
print('10 mã mô phỏng; 8 quan tâm Data Analyst; điểm thô 4/5, làm trơn =', round(da.rating,4))

,role_code,demand_count,actors,vote_count,vote_average,rating
0,data_analyst,8.0,8.0,10,4.0,2.75
5,backend,2.0,2.0,10,1.0,0.75


10 mã mô phỏng; 8 quan tâm Data Analyst; điểm thô 4/5, làm trơn = 2.75


## 3. Kết hợp điểm nội dung với rating

Chỉ cộng rating cho ứng viên đã đạt ngưỡng nội dung/vùng. Không đủ 5 trình duyệt:
giữ nguyên điểm. Khi đủ: `0.90*content + 0.10*WR/5`. Không cộng hai đại lượng
khác thang đo trực tiếp. Các trọng số cần đánh giá, không mặc định tăng độ chính xác.

In [4]:
# Điểm nội dung giả lập để kiểm tra công thức, ghi rõ không phải model score.
demo_jobs = [dict(id='DEMO_A', role_code='data_analyst', score=0.8),
             dict(id='DEMO_B', role_code='backend', score=0.82)]
display(pd.DataFrame(DemandRatingStore.rerank(demo_jobs, snapshot)))
assert len(DemandRatingStore.rerank([],snapshot)) == 0

,id,role_code,score,content_score,demand_rating,demand_count,demand_actors,demand_weight,personal_score
0,DEMO_A,data_analyst,0.80,0.80,2.75,8.0,10,0.1,0.775
1,DEMO_B,backend,0.82,0.82,0.75,2.0,10,0.1,0.753


## 4. Hai lựa chọn địa điểm trên dữ liệu thật

Hybrid thật, tắt semantic trong notebook để chạy CPU nhẹ. Web có thể bật model.
CV ví dụ: **ở TP.HCM, muốn làm Hà Nội**. Nơi ở và nơi muốn làm là hai trường riêng.
Đối chiếu toàn quốc không thay thế danh sách chính đã lọc Hà Nội.

In [5]:
cv_text = """Vị trí ứng tuyển: Backend Developer Python
Kỹ năng: Python, SQL, Django, FastAPI, PostgreSQL, Docker, Git
Kinh nghiệm: 2 năm
Địa chỉ: TP.HCM
Nơi muốn làm việc: Hà Nội"""
engine = HybridJobRecommenderSystem(localize_jobs(jobs,'vi'), use_semantic=False)
home = home_locations(cv_text)[0]
regional = engine.match_cv_to_jobs(cv_text,5,location='ha_noi')
assert all('ha_noi' in values.split(';') for values in regional.Region_Codes)
display(regional[['Job_ID','Title','Region_Codes','Similarity_Score']])
national = engine.match_cv_to_jobs(cv_text,len(jobs))
records = [dict(id=row.Job_ID, title=row.Title, score=float(row.Similarity_Score),
                regions=str(row.Region_Codes).split(';')) for row in national.itertuples()]
comparison = compare_options(records,home)
print('Chính sách kết hợp (thử nghiệm):', comparison['balance_policy'])
for key in ['balanced','best_fit','near_home']:
    print(key)
    table = pd.DataFrame(comparison[key])
    columns = ['id','title','score','proximity_score','balanced_score','regions','distance_km','same_region']
    display(table[[column for column in columns if column in table.columns]])

,Job_ID,Title,Region_Codes,Similarity_Score
0,VN6682,Python Backend Developer,ha_noi,0.707357
1,VN0117,Backend Developer - Senior Level (Pythonl) | T...,ha_noi,0.684400
2,VN7045,Senior Backend Developer (Python),ha_noi,0.674789
3,VN0929,Backend Developer,ha_noi,0.660281
4,VN5825,Backend Developer (Python/Django),ha_noi,0.654155


Chính sách kết hợp (thử nghiệm): {'fit_weight': 0.7, 'proximity_weight': 0.3, 'distance_scale_km': 50.0, 'version': 'career-distance-v1'}
balanced


,id,title,score,proximity_score,balanced_score,regions,distance_km,same_region
0,VN5489,Python Backend Developer,0.714213,1.0,0.799949,[ho_chi_minh],0,True
1,VN0324,Senior Backend Developer,0.696299,1.0,0.787409,[ho_chi_minh],0,True
2,VN5952,Intern Backend Python (Django REST Framework),0.673514,1.0,0.771460,[ho_chi_minh],0,True


best_fit


,id,title,score,regions,distance_km,same_region
0,VN5489,Python Backend Developer,0.714213,[ho_chi_minh],0,True
1,VN6682,Python Backend Developer,0.707357,[ha_noi],1137,False
2,VN0324,Senior Backend Developer,0.696299,[ho_chi_minh],0,True


near_home


,id,title,score,regions,distance_km,same_region
0,VN5489,Python Backend Developer,0.714213,[ho_chi_minh],0,True
1,VN0324,Senior Backend Developer,0.696299,[ho_chi_minh],0,True
2,VN5952,Intern Backend Python (Django REST Framework),0.673514,[ho_chi_minh],0,True


Khoảng cách là Haversine giữa điểm đại diện vùng GeoNames, **không phải đường
đi từ nhà đến công ty**. Cùng vùng không có nghĩa đi làm 0km. Chưa đọc được nơi
ở phải xác nhận, không lấy nơi muốn làm để thay thế.

## 5. Phạm vi đánh giá

Kết quả A/B/C trong `outputs/evaluation` thuộc **snapshot500** đóng băng ở
`data/evaluation/jobs_catalog_500.csv`. Không dùng những chỉ số đó như đánh giá
cho catalog7050 hoặc cho rating/khoảng cách mới. Cần qrels mới, người gán nhãn
độc lập và ablation cùng tập truy vấn để kiểm chứng cải thiện.